In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.datasets import ImageFolder
import timm

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

device

'cuda'

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")

print("Path to dataset files:", path)

100%|██████████| 200M/200M [00:01<00:00, 163MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/fahadullaha/facial-emotion-recognition-dataset/versions/1


In [ ]:
class FacesImagesDataset(Dataset):
  def __init__(self, data_dir, transform=None):
    self.data = ImageFolder(root=data_dir, transform=transform)

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx]

  @property
  def classes(self):
    return self.data.classes

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_dataset = FacesImagesDataset(data_dir=f'{path}/processed_data',
                                  transform=transform)

In [ ]:
targets = full_dataset.data.targets

train_idx, temp_idx, train_targets, temp_targets = train_test_split(
    range(len(targets)),
    targets,
    test_size=0.3,
    stratify=targets,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_targets,
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

In [ ]:
train_dataset[0][0].shape

torch.Size([3, 224, 224])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
timm.list_models('*resnet*')

['cspresnet50',
 'cspresnet50d',
 'cspresnet50w',
 'eca_resnet33ts',
 'ecaresnet26t',
 'ecaresnet50d',
 'ecaresnet50d_pruned',
 'ecaresnet50t',
 'ecaresnet101d',
 'ecaresnet101d_pruned',
 'ecaresnet200d',
 'ecaresnet269d',
 'ecaresnetlight',
 'gcresnet33ts',
 'gcresnet50t',
 'inception_resnet_v2',
 'lambda_resnet26rpt_256',
 'lambda_resnet26t',
 'lambda_resnet50ts',
 'legacy_seresnet18',
 'legacy_seresnet34',
 'legacy_seresnet50',
 'legacy_seresnet101',
 'legacy_seresnet152',
 'nf_ecaresnet26',
 'nf_ecaresnet50',
 'nf_ecaresnet101',
 'nf_resnet26',
 'nf_resnet50',
 'nf_resnet101',
 'nf_seresnet26',
 'nf_seresnet50',
 'nf_seresnet101',
 'resnet10t',
 'resnet14t',
 'resnet18',
 'resnet18d',
 'resnet26',
 'resnet26d',
 'resnet26t',
 'resnet32ts',
 'resnet33ts',
 'resnet34',
 'resnet34d',
 'resnet50',
 'resnet50_clip',
 'resnet50_clip_gap',
 'resnet50_gn',
 'resnet50_mlp',
 'resnet50c',
 'resnet50d',
 'resnet50s',
 'resnet50t',
 'resnet50x4_clip',
 'resnet50x4_clip_gap',
 'resnet50x16_clip

In [ ]:
class FacialEmotionClassifier(nn.Module):
  def __init__(self):
    super(FacialEmotionClassifier, self).__init__()

    self.base_model = timm.create_model('resnet152', pretrained=True)

    self.features = nn.Sequential(*list(self.base_model.children())[:-1])

    for param in self.features.parameters():
      param.requires_grad = False

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 7)
    )

  def forward(self, x):
    x = self.features(x)
    output = self.classifier(x)
    return output

In [ ]:
epochs = 10

model = FacialEmotionClassifier()
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_val_loss = -1

print(device)
for epoch in range(epochs):

  if epoch == 4:
    for param in model.features.parameters():
      param.requires_grad = True
      optimizer = optim.Adam(model.parameters(), lr=0.0001)

  model.train()
  running_loss = 0.0
  for images, labels in tqdm(train_loader, desc='Training Batches'):
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    pred = model(images)
    loss = criterion(pred, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item() * labels.size(0)

  train_loss = running_loss / len(train_loader.dataset)

  model.eval()
  running_loss = 0.0
  with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Validation Batches'):
      images, labels = images.to(device), labels.to(device)
      pred = model(images)
      loss = criterion(pred, labels)
      running_loss += loss.item() * labels.size(0)

  val_loss = running_loss / len(val_loader.dataset)

  if epoch == 0:
    best_val_loss = val_loss

  if val_loss <= best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), 'FacialEmotionClassifier.pt')
  print(f'Epoch {epoch + 1}/{epochs} - Training Loss : {train_loss} - Validation Loss : {val_loss}')

cuda


Training Batches:   0%|          | 0/1089 [00:00<?, ?it/s]

Validation Batches:   0%|          | 0/234 [00:00<?, ?it/s]

Epoch 1/10 - Training Loss : 1.560653407492619 - Validation Loss : 1.4612874392606143


Training Batches:   0%|          | 0/1089 [00:00<?, ?it/s]

...

Validation Batches:   0%|          | 0/234 [00:00<?, ?it/s]

Epoch 9/10 - Training Loss : 0.4568012056280233 - Validation Loss : 0.8408909343213973


Training Batches:   0%|          | 0/1089 [00:00<?, ?it/s]

Validation Batches:   0%|          | 0/234 [00:00<?, ?it/s]

Epoch 10/10 - Training Loss : 0.3334573748216411 - Validation Loss : 0.9486873704170277


In [ ]:
from google.colab import drive

# 1. ربط الـ Drive بالـ Colab (هيطالبك بصلاحيات الوصول)
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# 2. نسخ الملف لمجلد Drive الرئيسي
import shutil

shutil.copy("FacialEmotionClassifier.pt", "/content/drive/MyDrive/FacialEmotionClassifier.pt")
print(" تم نسخ الموديل للـ Drive بنجاح!")

Mounted at /content/drive
 تم نسخ الموديل للـ Drive بنجاح!


In [ ]:
import shutil

shutil.copy("/content/drive/MyDrive/FacialEmotionClassifier.pt", "FacialEmotionClassifier.pt")

'FacialEmotionClassifier.pt'

In [ ]:
model = FacialEmotionClassifier()

model.load_state_dict(torch.load("FacialEmotionClassifier.pt", map_location=device))

model.to(device)

FacialEmotionClassifier(
  (base_model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act1): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act2): ReLU(inplace=True)
        (aa): Identity()
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(2

In [ ]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
  for images, labels in tqdm(test_loader, desc='Test Batches'):
    images, labels = images.to(device), labels.to(device)
    predictions = model(images)
    _ , pred = torch.max(predictions, 1)
    correct += (labels == pred).sum().item()
    total += labels.size(0)

accuracy = 100 * correct / total
print(f'Accuracy = {accuracy}')

Test Batches:   0%|          | 0/234 [00:00<?, ?it/s]

Accuracy = 70.40310700415161
